In [18]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
import datetime as dt
import Engine as Ctool
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import math
from scipy.optimize import minimize as minimize_cpu
import jax.numpy as jnp
from jax.scipy.optimize import minimize as minimize_gpu

In [19]:
date = dt.date(2026, 4, 1)
index = "SP500"
lbklen = "-30b"
end_date = date
start_date = Ctool.calc_add_date(end_date, lbklen)

In [20]:
universe = Ctool.load_universe(index, date)[index]
alphas = {}
data = Ctool.calc_dict2df(Ctool.calc_forward_close(universe, start_date, end_date))
for ticker in universe:
    data = Ctool.calc_forward_close(ticker, start_date, end_date)[ticker]
    data['value'] = data['adjusted_close'] - data['adjusted_close'].rolling(20).mean()
    data['mean'] = data['value'].rolling(5).mean()
    data['std'] = data['value'].rolling(5).std()
    data['alpha'] = (data['value'] - data['mean']) / data['std']
    if not np.isnan(data['alpha'].iloc[-1]):
        alphas[ticker] = float(data['alpha'].iloc[-1])
keys, values = alphas.keys(), Ctool.calc_zscore(list(alphas.values()))
alphas = dict(zip(keys, values))

In [21]:
def backtest_return(date, alphas, rf=0, lbklen_std='20b'):
    def objective(weights, returns):
        ret = returns.reshape(1, -1) @ weights.reshape(-1, 1)
        std = weights.reshape(1, -1) @ cov.values @ weights.reshape(-1, 1)
        return - ((ret - rf) / std) * math.sqrt(252)
    tickers = list(alphas.keys())
    start_date = Ctool.calc_add_date(date, "-"+lbklen_std)
    end_date = Ctool.calc_add_date(date, "-1b")
    cov = Ctool.calc_dict2df(Ctool.load_return(tickers, start_date, end_date, "1b")).dropna(axis=1, how='any').cov()
    tickers = list(set(tickers)&set(cov.columns.tolist()))
    alphas = np.array([alphas[ticker] for ticker in tickers])
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(len(tickers)))
    init_guess = len(tickers) * [1. / len(tickers)]
    result = minimize_cpu(objective, init_guess, args=(alphas), method='SLSQP', bounds=bounds, constraints=constraints)
    returns = Ctool.calc_dict2df(Ctool.load_return(tickers, date, date, '1b')).values
    ret = (returns @ result.x.reshape(-1,)).item()
    res = -objective(result.x, returns)
    position = dict(zip(tickers, result.x.tolist()))
    return (ret, res, position)

In [16]:
def backtest_return_gpu(date, alphas, rf=0, lbklen_std='20b'):
    def objective(weights, returns):
        ret = returns.reshape(1, -1) @ weights.reshape(-1, 1)
        std = weights.reshape(1, -1) @ cov.values @ weights.reshape(-1, 1)
        return - ((ret - rf) / std) * math.sqrt(252)
    tickers = list(alphas.keys())
    start_date = Ctool.calc_add_date(date, "-"+lbklen_std)
    end_date = Ctool.calc_add_date(date, "-1b")
    cov = Ctool.calc_dict2df(Ctool.load_return(tickers, start_date, end_date, "1b")).dropna(axis=1, how='any').cov()
    tickers = list(set(tickers)&set(cov.columns.tolist()))
    alphas = jax.numpy.asarry([alphas[ticker] for ticker in tickers])
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for _ in range(len(tickers)))
    init_guess = len(tickers) * [1. / len(tickers)]
    result = minimize(objective, init_guess, args=(alphas), method='SLSQP', bounds=bounds, constraints=constraints)
    returns = Ctool.calc_dict2df(Ctool.load_return(tickers, date, date, '1b')).values
    ret = (returns @ result.x.reshape(-1,)).item()
    res = -objective(result.x, returns)
    position = dict(zip(tickers, result.x.tolist()))
    return (ret, res, position)

In [22]:
ret, res, position = backtest_return(date, alphas)

In [ ]:
p